# Kaggle runner — exp_0019 / exp_0020 (TabM baseline)

Trains the second neural family on GPU, on the **frozen 5-fold partition** shipped as
the `s6e7-frozen-folds` dataset (never rebuilt — rule 6):

- **exp_0019** — TabM (k=32, piecewise-linear embeddings, balanced cross-entropy),
  13 raw features. One variable vs exp_0004: the model family.
- **exp_0020** — same + the 9 exact-value target-encoding columns of exp_0017
  (`encoders.exact_value_te`, fitted inside each fold). One variable vs exp_0019.

Class weights are balanced *in training*, so the ledger's argmax `cv_mean` is already
the prior-corrected number — directly comparable with exp_0004 (0.94956) and exp_0017
(0.95014). No rule run needed. All logic imports from `src/`; this notebook only
orchestrates and displays.

**Carry back** (`kaggle kernels output`): `artifacts/exp_00{19,20}{,_test}.npy` →
`oof/`, and the two new rows of `artifacts/experiments.csv` → the local ledger.

In [ ]:
# Diagnostics first, stdlib only. Kaggle's log endpoint has returned empty files for
# errored runs, so this cell makes the run self-reporting: mounts go to an output file,
# and any later cell's exception is written to diag_error.txt before it propagates
# (output files survive an errored run).
import json
import sys
import traceback
from pathlib import Path

diag = {
    "python": sys.version,
    "inputs": {
        p.name: sorted(f.name for f in p.iterdir())[:10]
        for p in Path("/kaggle/input").iterdir()
    },
}
Path("/kaggle/working/diag_mounts.json").write_text(json.dumps(diag, indent=2))
print(json.dumps(diag, indent=2))


def _dump_exc(shell, etype, evalue, tb, tb_offset=None):
    text = "".join(traceback.format_exception(etype, evalue, tb))
    Path("/kaggle/working/diag_error.txt").write_text(text)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


get_ipython().set_custom_exc((BaseException,), _dump_exc)

In [ ]:
# --no-deps is load-bearing: a plain install can let pip replace the image's torch with a
# PyPI wheel lacking kernels for the assigned GPU (the FTT runner died on exactly that).
# tabm's only other dependencies (torch, typing_extensions) ship with the image.
%pip install -q --no-deps tabm==0.0.3 rtdl_num_embeddings==0.0.12

In [ ]:
import shutil
import subprocess

# The repo lives in /tmp, NOT /kaggle/working: only artifacts/ and diag files should
# land in the output snapshot, so a failed run stays kilobytes and pulls in seconds.
REPO = Path("/tmp/repo")

clone = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/epsilonlog/comp-playground-series-s6e7.git", str(REPO)],
    capture_output=True, text=True,
)
assert (REPO / "src").exists(), f"clone failed: {clone.stderr}"

inputs = Path("/kaggle/input")


def first_existing(*candidates: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"none exist: {[str(c) for c in candidates]}")


# Mount layout differs across Kaggle images: classic /kaggle/input/<slug> vs the
# nested /kaggle/input/competitions/<slug> and /kaggle/input/datasets/<owner>/<slug>.
comp = first_existing(
    inputs / "competitions" / "playground-series-s6e7",
    inputs / "playground-series-s6e7",
)
ds = first_existing(
    inputs / "datasets" / "aligh474" / "s6e7-frozen-folds",
    inputs / "s6e7-frozen-folds",
)

raw = REPO / "data" / "raw"
processed = REPO / "data" / "processed"
raw.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)

for n in ["train.csv", "test.csv", "sample_submission.csv"]:
    src = comp / n if (comp / n).exists() else ds / n
    assert src.exists(), f"{n} found in neither {comp} nor {ds}"
    shutil.copy(src, raw / n)
shutil.copy(ds / "folds.parquet", processed / "folds.parquet")

print("csv source:", comp)
print("raw:", sorted(p.name for p in raw.iterdir()))
print("processed:", sorted(p.name for p in processed.iterdir()))

In [ ]:
import sys

sys.path.insert(0, "/tmp/repo/src")

import numpy as np
import rtdl_num_embeddings
import tabm
import torch

from s6e7 import cv, folds, io
from s6e7.cv import ExperimentConfig

print("tabm", tabm.__version__, "| rtdl_num_embeddings", rtdl_num_embeddings.__version__,
      "| torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| bf16:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())
assert torch.cuda.is_available(), "no GPU - check the kernel's accelerator setting"
train, test = io.load_train(), io.load_test()
folds.verify(train)  # the shipped file IS the frozen partition; prove it arrived intact
print("folds verified:", folds.FOLDS_PATH)

Smoke first: does the wrapper run end to end on GPU, with and without the encoder?
34k rows, 2 epochs, never logged. The scores are meaningless **by design** — this cell
only proves the plumbing before an hour of training.

In [ ]:
smoke = train.head(34_000)
for encoder in ("", "exact_value_te"):
    r = cv.run(
        ExperimentConfig(exp_id="smoke_tabm", model="tabm",
                         params={"max_epochs": 2, "verbose": False}, encoder=encoder),
        train=smoke,
        log=False,
    )
    label = "tabm + " + encoder if encoder else "tabm"
    print(f"{label}: plumbing ok, {r.runtime_s:.0f}s (scores meaningless at this n)")

Full runs. Each fold prints one line per epoch (holdout NLL and balanced accuracy on the
inner 10% holdout, `*` marks a new best) — the early-stopping trace is the only place the
model's training dynamics are visible, so it is kept in the log on purpose.

In [ ]:
result_19 = cv.run(
    ExperimentConfig(
        exp_id="exp_0019",
        model="tabm",
        parent="exp_0004",
        changed="new family: TabM (k=32, piecewise-linear embeddings, balanced CE), raw features",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0019  cv_mean={result_19.cv_mean:.5f}  cv_std={result_19.cv_std:.5f}  runtime={result_19.runtime_s:.0f}s")
print("fold scores:", [round(s, 5) for s in result_19.fold_scores])
print("fit scores: ", [round(s, 5) for s in result_19.fit_scores], "(fit - val gap = the over/underfitting dial)")

In [ ]:
result_20 = cv.run(
    ExperimentConfig(
        exp_id="exp_0020",
        model="tabm",
        encoder="exact_value_te",
        parent="exp_0019",
        changed="add 9 fold-fitted exact-value TE columns (exp_0017's encoder)",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0020  cv_mean={result_20.cv_mean:.5f}  cv_std={result_20.cv_std:.5f}  runtime={result_20.runtime_s:.0f}s")
print("fold scores:", [round(s, 5) for s in result_20.fold_scores])
print("fit scores: ", [round(s, 5) for s in result_20.fit_scores], "(fit - val gap = the over/underfitting dial)")

In [ ]:
art = Path("/kaggle/working/artifacts")
art.mkdir(exist_ok=True)
for rel in ["oof/exp_0019.npy", "oof/exp_0019_test.npy", "oof/exp_0020.npy", "oof/exp_0020_test.npy", "experiments.csv"]:
    shutil.copy(REPO / rel, art / Path(rel).name)
print(sorted(f"{p.name} ({p.stat().st_size:,}B)" for p in art.iterdir()))